# LLM Zoomcamp 2026 - Homework 2: Vector Search


## Setup

Load the helper libraries, embedding model, and course lesson documents.

In [3]:
import numpy as np
from gitsource import GithubRepositoryDataReader

from embedder import Embedder


embedder = Embedder()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

len(documents), documents[0].keys(), documents[0]["filename"]

(72, dict_keys(['content', 'filename']), '01-agentic-rag/lessons/01-intro.md')

## Q1. Embedding A Query

Goal: generate an embedding for the homework query and inspect its values.

In [4]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

v.shape, v[0]

((384,), np.float64(-0.02058203437252893))

## Q2. Cosine Similarity

Goal: compute cosine similarity between embeddings to compare semantic closeness.

In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_filename)

doc_vector = embedder.encode(target_doc["content"])
similarity = doc_vector.dot(v)

similarity

np.float64(0.36107026789538205)

## Q3. Chunking And Manual Vector Search

Goal: split documents into chunks and search them manually with vector similarity.

In [6]:
from gitsource import chunk_documents


chunks = chunk_documents(documents, size=2000, step=1000)

chunk_vectors = embedder.encode_batch([chunk["content"] for chunk in chunks])
X = np.vstack(chunk_vectors)

scores = X.dot(v)
best_idx = scores.argmax()
best_chunk = chunks[best_idx]

len(chunks), scores[best_idx], best_chunk["filename"], best_chunk["start"]

(295,
 np.float64(0.6489016436447387),
 '02-vector-search/lessons/07-sqlitesearch-vector.md',
 1000)

## Q4. Vector Search With Minsearch

Goal: build a minsearch vector index and retrieve relevant chunks.

In [7]:
from minsearch import VectorSearch


vector_index = VectorSearch(keyword_fields=["filename"]).fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

q4_results = vector_index.search(q4_vector, num_results=5)

[(result["filename"], result["start"]) for result in q4_results]

[('04-evaluation/lessons/05-search-metrics.md', 0),
 ('04-evaluation/lessons/01-intro.md', 0),
 ('01-agentic-rag/lessons/05-search.md', 0),
 ('04-evaluation/lessons/01-intro.md', 2000),
 ('04-evaluation/lessons/15-next-steps.md', 0)]

## Q5. Text Search Vs Vector Search

Goal: compare text-based search results with vector search results.

In [8]:
from minsearch import Index


text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

q5_vector_results = vector_index.search(q5_vector, num_results=5)
q5_text_results = text_index.search(q5_query, num_results=5)

vector_files = {result["filename"] for result in q5_vector_results}
text_files = {result["filename"] for result in q5_text_results}

[(result["filename"], result["start"]) for result in q5_vector_results], [(result["filename"], result["start"]) for result in q5_text_results], vector_files - text_files

([('02-vector-search/lessons/08-pgvector.md', 0),
  ('02-vector-search/lessons/08-pgvector.md', 3000),
  ('03-orchestration/lessons/05-rag.md', 2000),
  ('02-vector-search/lessons/08-pgvector.md', 1000),
  ('02-vector-search/lessons/08-pgvector.md', 2000)],
 [('02-vector-search/lessons/02-embeddings.md', 4000),
  ('03-orchestration/lessons/05-rag.md', 1000),
  ('02-vector-search/lessons/01-intro.md', 0),
  ('03-orchestration/lessons/05-rag.md', 0),
  ('02-vector-search/lessons/01-intro.md', 1000)],
 {'02-vector-search/lessons/08-pgvector.md'})

## Q6. Hybrid Search With RRF

Goal: combine text and vector search rankings using reciprocal rank fusion.

In [9]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

q6_vector_results = vector_index.search(q6_vector, num_results=5)
q6_text_results = text_index.search(q6_query, num_results=5)
q6_results = rrf([q6_vector_results, q6_text_results])

[(result["filename"], result["start"]) for result in q6_results]

[('01-agentic-rag/lessons/13-function-calling.md', 4000),
 ('01-agentic-rag/lessons/01-intro.md', 2000),
 ('01-agentic-rag/lessons/14-agentic-loop.md', 0),
 ('04-evaluation/lessons/02-ground-truth.md', 1000),
 ('01-agentic-rag/lessons/16-other-frameworks.md', 0)]

## Final Answers

Based on this run:

- Q1: `-0.02`
- Q2: `0.37`
- Q3: `02-vector-search/lessons/07-sqlitesearch-vector.md`
- Q4: `04-evaluation/lessons/05-search-metrics.md`
- Q5: `02-vector-search/lessons/08-pgvector.md`
- Q6: `01-agentic-rag/lessons/13-function-calling.md`